[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github&logoColor=white)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-H3_Colab.ipynb)


# 🎬 MiniMax-H3 — Video + Audio Generation (33B, INT4 quantized, MIT-adjacent)

A Colab port of [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) — a 33B parameter video generation model that produces **video with synchronized audio** (ambience, foley, speech). Supports text-to-video, image-to-video (first/last frame), and reference-based generation.

## How it works

MiniMax-H3 uses a **split deployment** architecture:

1. **Remote conditioner** (via `gradio_client` → `multimodalart/qwen3vl-conditioner` HF Space): encodes the text prompt + optional keyframe images into `prompt_embeds` + `text_token_tags`. This runs on HuggingFace's ZeroGPU — no local text encoder needed (saves 62 GB download + 14 GB VRAM).

2. **Local denoiser** (on your Colab GPU): the 33B transformer (DiT) denoises the video+audio latents, then the VAEs decode them into frames + stereo audio. The transformer is **INT4-quantized on-the-fly** with torchao (~15.5 GB VRAM), and VAEs are loaded sequentially after denoising (~10.3 GB). Peak VRAM: ~18.5 GB — fits on L4!

```
prompt + images → [remote conditioner] → prompt_embeds
                                              ↓
              INT4 DiT (15.5 GB) → denoise → latents → VAEs → video.mp4 + audio
```

## ⚠️ License — Territory Restriction + MAU Cap

MiniMax H3 Community License:
- **Excludes:** EU, UK, South Korea, **and USA**
- **>1M MAU** requires separate commercial license
- Continuing past the header cell is your acceptance of the license

## Quick start

1. **Runtime → Change runtime type → GPU** (L4 or A100 recommended)
2. Run **STEP 1** — installs torch 2.11.0+cu128, diffusers from PR commit, torchao, transformers, etc. First run: ~10-15 min.
3. Run **STEP 2** — downloads FL2VA transformer (61.7 GB) + VAEs (10.3 GB) to Drive cache. First run: ~30-60 min depending on network.
4. Run **STEP 3** — imports, stub `spaces` module, lazy model loader with torchao INT4 quantization
5. Run **STEP 4** — opens the Gradio UI. Enter a prompt, pick canvas/duration/steps, click Generate.
6. **STEP 5** keep-alive, **STEP 6** quick test, **STEP 7** batch

## Outputs

```
output.mp4    # Video + synchronized stereo audio (32 kHz)
```

## Requirements

* **GPU**: L4 (22 GB) or A100 (40 GB). T4 (15 GB) is too small for INT4.
* **Disk**: ~72 GB for FL2VA weights (cached on Drive after first download)
* **First-run**: ~45-75 min total (download + install + quantization)
* **Subsequent**: ~5-10 min (load from Drive cache + quantize)
* **Inference**: ~2-10 min per video depending on canvas/frames/steps
* **Network**: Requires access to `multimodalart/qwen3vl-conditioner` HF Space (public, no token needed)

## Technical notes

- **torch 2.11.0+cu128**: The diffusers PR requires torch>=2.6. We pin 2.11.0+cu128 (matches Colab's CUDA 12.8).
- **diffusers from PR commit 665f578**: MiniMax-H3 support is in an open WIP PR, not on PyPI. We install from the specific commit.
- **torchao INT4**: The 62 GB bf16 transformer is quantized to INT4 on-the-fly (~15.5 GB VRAM). This adds ~5 min to startup but makes the model fit on L4.
- **Remote conditioner**: The 62 GB Qwen3-VL text encoder runs on the HF Space `multimodalart/qwen3vl-conditioner`. We call it via `gradio_client`. This saves 62 GB download + 14 GB VRAM.
- **`spaces` stub**: The upstream code uses `import spaces` for HF ZeroGPU. We install a stub module so the code runs on Colab.
- **VAEs**: Video VAE (fp16, ~4.85 GB) + Audio VAE (fp32, ~577 MB). Audio VAE must stay fp32 (bf16 decodes 20 dB too quiet).

## Companion notebooks

- **Wan2.2_Colab** — text/image-to-video (no audio)
- **Wan2.2_Animate_Colab** — character animation
- **Wan2.2_S2V_Colab** — sound-to-video
- **GaussianGPT_Colab** — autoregressive 3D Gaussian scene generation
- **InfiniSplat_Colab** — single-image 3DGS reconstruction


In [ ]:
#@title STEP 1 — Install torch 2.11.0+cu128, diffusers PR, torchao, transformers, av
"""
• Pins torch to 2.11.0+cu128 (diffusers PR requires torch>=2.6; we use 2.11 for best compat)
• Installs diffusers from the MiniMax-H3 WIP PR (commit 665f578, not on PyPI)
• Installs torchao for INT4 quantization
• Pins transformers 5.8.0 (required by the diffusers PR for Qwen3-VL processor)
• Installs PyAV for video+audio muxing
• Stubs the `spaces` module (HF ZeroGPU API, not available on Colab)
"""
import os, sys, time, subprocess, pathlib, types

print('='*72)
print('MiniMax-H3 — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MiniMax-H3')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Drive cache  : {drive_root}')
else:
    drive_root = pathlib.Path('/content/_h3_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}')

OUT_DIR = drive_root / 'h3_out'
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT = pathlib.Path('/content/h3_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

t_total = time.time()

# 1. Pin torch to 2.11.0+cu128 ───────────────────────────────────────────
TARGET_TORCH = '2.11.0'
if not torch.__version__.startswith(TARGET_TORCH):
    print(f'\n[1/5] Pinning torch to {TARGET_TORCH}+cu128 ...')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--disable-pip-version-check', '--no-input',
        f'torch=={TARGET_TORCH}+cu128',
        'torchvision==0.26.0+cu128',
        'torchaudio==2.11.0+cu128',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
        '--force-reinstall',
    ], check=False)
    print(f'  torch pinned in {time.time()-t0:.1f}s')
    import importlib
    importlib.reload(importlib.import_module('torch'))
    import torch
    print(f'  torch now    : {torch.__version__}  (CUDA {torch.version.cuda})')
else:
    print(f'\n[1/5] torch {torch.__version__} already matches {TARGET_TORCH} — skipping')

# 2. Install diffusers from the MiniMax-H3 PR ───────────────────────────
print('\n[2/5] Installing diffusers from MiniMax-H3 PR (commit 665f578) ...')
t0 = time.time()
DIFFUSERS_PR_COMMIT = '665f578278365ea4a3318cb8c9b66ce6c01204b9'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    f'git+https://github.com/huggingface/diffusers.git@{DIFFUSERS_PR_COMMIT}',
], check=False)
print(f'  diffusers installed in {time.time()-t0:.1f}s')

# 3. Install torchao + transformers + accelerate ────────────────────────
print('\n[3/5] Installing torchao + transformers + accelerate ...')
t0 = time.time()
EXTRA_PKGS = [
    'torchao>=0.7.0',
    'transformers==5.8.0',
    'accelerate==1.14.0',
    'huggingface-hub>=1.24.0',
    'safetensors>=0.8.0',
    'av',
    'einops',
    'omegaconf',
    'gradio>=5.49.1,<7',
    'pillow',
    'numpy<2.0',
    'scipy',
    'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_PKGS, check=False)
print(f'  Extra deps installed in {time.time()-t0:.1f}s')

# 4. Stub the `spaces` module (HF ZeroGPU, not available on Colab) ─────
print('\n[4/5] Stubbing `spaces` module ...')
spaces_stub = types.ModuleType('spaces')
def _gpu_decorator(duration=120, size=None):
    def decorator(fn):
        return fn
    return decorator
spaces_stub.GPU = _gpu_decorator
sys.modules['spaces'] = spaces_stub
print('  spaces stub installed')

# 5. Verify imports ─────────────────────────────────────────────────────
print('\n[5/5] Verifying imports ...')
t0 = time.time()
try:
    import diffusers
    print(f'  diffusers  : {diffusers.__version__}')
except ImportError as e:
    print(f'  [FAIL] diffusers: {e}')
try:
    import torchao
    print(f'  torchao    : {torchao.__version__}')
except ImportError as e:
    print(f'  [WARN] torchao: {e}')
try:
    import transformers
    print(f'  transformers: {transformers.__version__}')
except ImportError as e:
    print(f'  [FAIL] transformers: {e}')
try:
    import av
    print(f'  av         : OK')
except ImportError as e:
    print(f'  [FAIL] av: {e}')

elapsed = time.time() - t_total
print()
print('='*72)
print(f'STEP 1 complete in {elapsed/60:.1f} min')
print('='*72)
print(f'  Drive cache  : {drive_root}')
print(f'  Output dir   : {OUT_DIR}')
print()
print('Next: run STEP 2 (download FL2VA weights).')


In [ ]:
#@title STEP 2 — Download FL2VA transformer + VAEs to Drive cache
"""
Downloads only the denoiser half of MiniMax-H3:
  - FL2VA/transformer/ (13 shards, ~61.7 GB)
  - FL2VA/video_vae/source/model.safetensors (~9.7 GB)
  - FL2VA/audio_vae/model.safetensors (~577 MB)
  - FL2VA/config files (model_index.json, processor configs, etc.)

Total: ~72 GB. The text encoder (62 GB) is NOT downloaded — we use the
remote conditioner HF Space instead.

All files are cached on Drive so subsequent runs skip the download.
"""
import os, sys, time, pathlib
from huggingface_hub import snapshot_download

print('='*72)
print('MiniMax-H3 — Download FL2VA denoiser weights')
print('='*72)

MODEL_REPO = 'MiniMaxAI/MiniMax-H3'
CKPT_DIR = drive_root / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'  Model repo  : {MODEL_REPO}')
print(f'  Cache dir   : {CKPT_DIR}')
print()

# Download only the FL2VA denoiser components (no text_encoder)
ALLOW_PATTERNS = [
    'FL2VA/transformer/*.safetensors',
    'FL2VA/transformer/config.json',
    'FL2VA/transformer/model.safetensors.index.json',
    'FL2VA/video_vae/source/model.safetensors',
    'FL2VA/video_vae/source/config.json',
    'FL2VA/video_vae/config.json',
    'FL2VA/video_vae/*.py',
    'FL2VA/audio_vae/model.safetensors',
    'FL2VA/audio_vae/config.json',
    'FL2VA/audio_vae/config.yaml',
    'FL2VA/audio_vae/*.py',
    'FL2VA/audio_vae/metadata.json',
    'FL2VA/model_index.json',
    'FL2VA/processor/*.json',
    'FL2VA/tokenizer/*.json',
    'FL2VA/tokenizer/*.txt',
]

t_total = time.time()
print(f'  Downloading {len(ALLOW_PATTERNS)} pattern groups ...')
print(f'  (This downloads ~72 GB. First run: 30-60 min depending on network.)')
print()

snapshot_download(
    repo_id=MODEL_REPO,
    allow_patterns=ALLOW_PATTERNS,
    local_dir=str(CKPT_DIR),
    cache_dir=os.environ.get('HF_HOME'),
    max_workers=4,
)

elapsed = time.time() - t_total

# Verify downloads
print()
print('  Downloaded files:')
total_size = 0
for f in sorted(CKPT_DIR.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        total_size += sz
        if sz > 1024**3:
            print(f'    {f.relative_to(CKPT_DIR):>55s}  {sz/1024**3:.2f} GB')
        elif sz > 1024**2:
            print(f'    {f.relative_to(CKPT_DIR):>55s}  {sz/1024**2:.1f} MB')

print()
print('='*72)
print(f'STEP 2 complete in {elapsed/60:.1f} min')
print(f'  Total downloaded: {total_size/1024**3:.1f} GB')
print(f'  Cache dir: {CKPT_DIR}')
print('='*72)
print()
print('Next: run STEP 3 (imports + lazy model loader + INT4 quantization).')


In [ ]:
#@title STEP 3 — Imports, spaces stub, remote conditioner, lazy model loader with INT4
"""
• Stubs the `spaces` module (HF ZeroGPU, not available on Colab)
• Defines `call_remote_conditioner()` — calls the HF Space via gradio_client
• Defines `load_minimax_h3()` — loads the FL2VA denoiser with torchao INT4
• Defines `generate_video()` — the full pipeline: condition → denoise → decode → mux
"""
import os, sys, time, gc, pathlib, types, traceback, tempfile
import torch

print('='*72)
print('MiniMax-H3 — Imports + lazy model loader')
print('='*72)

# --- Stub the `spaces` module (must come before any diffusers import) ──
_spaces_stub = types.ModuleType('spaces')
def _gpu_decorator(duration=120, size=None):
    def decorator(fn):
        return fn
    return decorator
_spaces_stub.GPU = _gpu_decorator
sys.modules['spaces'] = _spaces_stub

# --- Verify deps ────────────────────────────────────────────────────────
import diffusers
import transformers
try:
    import torchao
    print(f'  diffusers    : {diffusers.__version__}')
    print(f'  transformers  : {transformers.__version__}')
    print(f'  torchao      : {torchao.__version__}')
except ImportError as e:
    print(f'  [FAIL] {e}')
    raise
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
else:
    print('  WARNING: no GPU detected')
print()

CKPT_DIR = drive_root / 'checkpoints'
CONDITIONER_SPACE = 'multimodalart/qwen3vl-conditioner'
MODEL_REPO = 'MiniMaxAI/MiniMax-H3'

# --- Canvas definitions (shared with the conditioner Space) ────────────
CANVASES = {
    "960x544 · 16:9 fast": (544, 960),
    "1024x576 · 16:9 fast": (576, 1024),
    "1152x640 · 16:9": (640, 1152),
    "1280x704 · 16:9": (704, 1280),
    "1344x768 · 16:9 full": (768, 1344),
    "544x960 · 9:16 fast": (960, 544),
    "640x1152 · 9:16": (1152, 640),
    "768x1344 · 9:16 full": (1344, 768),
    "544x544 · 1:1 fast": (544, 544),
    "768x768 · 1:1 full": (768, 768),
    "768x576 · 4:3 fast": (576, 768),
    "1024x768 · 4:3 full": (768, 1024),
    "576x768 · 3:4 fast": (768, 576),
    "768x1024 · 3:4 full": (1024, 768),
    "1152x512 · 21:9 fast": (512, 1152),
    "1536x672 · 21:9 full": (672, 1536),
}
DEFAULT_CANVAS = "960x544 · 16:9 fast"
FPS, FRAMES_PER_CHUNK, LATENTS_PER_CHUNK = 24, 17, 5
MAX_UI_DURATION = 14

def snap_frames(seconds):
    """The frame count MiniMax-H3's video VAE can decode: the next 17*n + 5 at 24 fps."""
    frames = max(1, round(float(seconds) * FPS))
    while frames % FRAMES_PER_CHUNK != LATENTS_PER_CHUNK:
        frames += 1
    return frames

# --- Remote conditioner (via gradio_client) ────────────────────────────
def call_remote_conditioner(prompt, image_path=None, last_image_path=None,
                              canvas=DEFAULT_CANVAS, num_frames=29,
                              rewrite_prompt=False, verbose=True):
    """Call the remote conditioner HF Space to get prompt_embeds + text_token_tags.

    Returns: (prompt_embeds, text_token_tags, metadata_dict, plan_dict)
    """
    from gradio_client import Client, handle_file
    from safetensors import safe_open

    if verbose:
        print(f'  [conditioner] Connecting to {CONDITIONER_SPACE} ...')
    t0 = time.time()
    client = Client(CONDITIONER_SPACE)
    if verbose:
        print(f'  [conditioner] Connected in {time.time()-t0:.1f}s')

    height, width = CANVASES[canvas]
    if verbose:
        print(f'  [conditioner] Encoding prompt ({len(prompt)} chars) at {width}x{height} ...')

    # The conditioner Space's /encode API
    path, plan = client.predict(
        prompt=prompt,
        image_path=handle_file(image_path) if image_path else None,
        last_image_path=handle_file(last_image_path) if last_image_path else None,
        canvas=canvas,
        num_frames=int(num_frames),
        rewrite_prompt=rewrite_prompt,
        api_name='/encode',
    )

    with safe_open(path, framework='pt') as handle:
        metadata = handle.metadata()
        prompt_embeds = handle.get_tensor('prompt_embeds')
        text_token_tags = handle.get_tensor('text_token_tags')

    if verbose:
        print(f'  [conditioner] Done in {time.time()-t0:.1f}s ({plan.get("num_text_tokens", "?")} tokens)')

    return prompt_embeds, text_token_tags, metadata, plan

# --- Lazy model loader (singleton) ────────────────────────────────────
_PIPE = None
_DEVICE = None
_QUANTIZED = False

def load_minimax_h3(verbose=True, quantize=True):
    """Load the MiniMax-H3 denoiser pipeline from cached checkpoints.

    Args:
        quantize: If True, apply torchao int4_weight_only to the transformer.
    Returns: (pipe, device)
    """
    global _PIPE, _DEVICE, _QUANTIZED
    if _PIPE is not None:
        return _PIPE, _DEVICE

    from diffusers import ComponentsManager
    from h3_split_blocks import MiniMaxH3GeneratorBlocks

    _DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # The generator blocks only load transformer + VAEs (no text_encoder)
    manager = ComponentsManager()
    blocks = MiniMaxH3GeneratorBlocks()

    if verbose:
        print(f'  [gen] Loading {[c.name for c in blocks.expected_components]} ...')
    t0 = time.time()
    pipe = blocks.init_pipeline(str(CKPT_DIR), components_manager=manager, collection='h3')
    pipe.load_components(dtype=torch.bfloat16)
    pipe.transformer.set_attention_backend('_native_cudnn')
    if verbose:
        print(f'  [gen] Model loaded in {time.time()-t0:.1f}s')

    # Apply torchao INT4 quantization to the transformer
    if quantize and not _QUANTIZED:
        if verbose:
            print(f'  [gen] Applying torchao INT4 quantization to transformer ...')
        t0 = time.time()
        from torchao.quantization import int4_weight_only, quantize_
        quantize_(pipe.transformer, int4_weight_only())
        _QUANTIZED = True
        if verbose:
            n_params = sum(p.numel() for p in pipe.transformer.parameters())
            print(f'  [gen] INT4 quantization done in {time.time()-t0:.1f}s ({n_params/1e9:.1f}B params)')

    # Move transformer to GPU, keep VAEs on CPU (sequential offload)
    pipe.transformer.to(_DEVICE)
    if verbose:
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info()
            print(f'  [gen] VRAM after DiT load: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')

    _PIPE = pipe
    return _PIPE, _DEVICE

def free_minimax_h3():
    """Unload the model and free GPU memory."""
    global _PIPE, _DEVICE, _QUANTIZED
    if _PIPE is not None:
        del _PIPE
    _PIPE = None
    _DEVICE = None
    _QUANTIZED = False
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Full generation pipeline ─────────────────────────────────────────
def generate_video(prompt, image_path=None, last_image_path=None,
                     canvas=DEFAULT_CANVAS, duration=5, steps=28, seed=42,
                     rewrite_prompt=False, verbose=True):
    """Full pipeline: remote condition → denoise → decode → mux video+audio.

    Returns: path to the output .mp4 file
    """
    from PIL import Image, ImageOps
    from diffusers.utils import encode_video

    num_frames = snap_frames(duration)

    # Phase 1: Remote conditioning
    if verbose:
        print(f'\n  Phase 1: Remote conditioning ...')
    prompt_embeds, text_token_tags, metadata, plan = call_remote_conditioner(
        prompt=prompt,
        image_path=image_path,
        last_image_path=last_image_path if last_image_path else None,
        canvas=canvas,
        num_frames=num_frames,
        rewrite_prompt=rewrite_prompt,
        verbose=verbose,
    )
    height = int(metadata['height'])
    width = int(metadata['width'])
    num_frames = int(metadata['num_frames'])

    # Phase 2: Load model (if not already loaded)
    pipe, device = load_minimax_h3(verbose=verbose)

    # Prepare keyframes (EXIF-transposed, RGB — same as the conditioner does)
    def keyframe(path):
        return ImageOps.exif_transpose(Image.open(path)).convert('RGB') if path else None

    # Phase 3: Denoise + decode (VAEs loaded on demand)
    if verbose:
        print(f'\n  Phase 2: Denoising {steps} steps at {width}x{height}, {num_frames} frames ...')
    t0 = time.time()

    # Move VAEs to GPU for decode (they were on CPU)
    pipe.vae.to(device)
    pipe.audio_vae.to(device)

    state = pipe(
        prompt_embeds=prompt_embeds.to(device),
        text_token_tags=text_token_tags,
        image=keyframe(image_path),
        last_image=keyframe(last_image_path),
        height=height,
        width=width,
        num_frames=num_frames,
        num_inference_steps=int(steps),
        generator=torch.Generator('cpu').manual_seed(int(seed)),
    )
    videos = state.get('videos')
    audio = state.get('audio')
    sampling_rate = state.get('sampling_rate')
    generate_seconds = time.time() - t0
    if verbose:
        print(f'  Phase 2 done in {generate_seconds:.0f}s')

    # Move VAEs back to CPU to free VRAM
    pipe.vae.to('cpu')
    pipe.audio_vae.to('cpu')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Phase 4: Mux video + audio
    if verbose:
        print(f'\n  Phase 3: Muxing video + audio ...')
    frames = videos[0]
    audio_data = audio[0].cpu() if hasattr(audio[0], 'cpu') else audio[0]
    sr = sampling_rate if sampling_rate is not None else 32000

    out_dir = OUT_DIR / f'gen_{int(time.time())}'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = str(out_dir / 'output.mp4')
    encode_video(frames, fps=FPS, output_path=out_path,
                 audio=audio_data, audio_sample_rate=sr)
    if verbose:
        sz = os.path.getsize(out_path) / 1024 / 1024
        print(f'  Output: {out_path} ({sz:.1f} MB)')

    return out_path, {
        'width': width,
        'height': height,
        'num_frames': num_frames,
        'steps': int(steps),
        'seed': int(seed),
        'generate_seconds': generate_seconds,
        'conditioner_tokens': plan.get('num_text_tokens', 0),
        'refined_prompt': plan.get('refined_prompt'),
    }

def free_cuda(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            free, total = torch.cuda.mem_get_info()
            print(f'  [cuda] free={free/1024**3:.1f} GB / total={total/1024**3:.1f} GB')

print('STEP 3 complete — model loader + remote conditioner + generate_video ready.')
print('Next: run STEP 4 to open the Gradio UI.')


In [ ]:
#@title STEP 4 — Gradio UI (text-to-video + image-to-video with audio)
"""
• Two-column layout: left = controls, right = video output
• Prompt + optional first/last frame images
• Canvas selector (aspect ratios)
• Duration slider (2-14 seconds)
• Steps slider (10-40)
• Seed
• Prompt rewrite toggle (asks the conditioner to rewrite into MiniMax-H3's trained format)
"""
import os, sys, time, pathlib, traceback
import torch
import gradio as gr

CSS = """
#col-container   { margin: 0 auto; max-width: 1400px; }
#main-title h1   { font-size: 2.4em !important; }
"""

with gr.Blocks(css=CSS, delete_cache=(600, 600)) as demo:
    gr.Markdown(
        '# **MiniMax-H3 — Video + Audio Generation**',
        elem_id='main-title',
    )
    gr.Markdown(
        'Generate video with synchronized audio from a text prompt '
        '(optionally with first/last frame images). '
        'Powered by [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) (33B, INT4 quantized).'
    )

    with gr.Row(elem_id='col-container'):
        with gr.Column(scale=1, min_width=380):
            prompt = gr.Textbox(
                label='Prompt',
                lines=3,
                value='A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot',
                info='Describe the scene. Audio (ambience, foley, speech) is generated automatically.',
            )
            with gr.Row():
                image = gr.Image(label='First frame (optional)', type='filepath', height=200)
                last_image = gr.Image(label='Last frame (optional)', type='filepath', height=200)
            btn_generate = gr.Button('Generate video', variant='primary')
            with gr.Accordion('Advanced options', open=False):
                canvas = gr.Dropdown(
                    label='Canvas (aspect ratio)',
                    choices=list(CANVASES.keys()),
                    value=DEFAULT_CANVAS,
                    info='Resolution and aspect ratio. "fast" variants are smaller and quicker.',
                )
                duration = gr.Slider(
                    2, MAX_UI_DURATION, value=5, step=1,
                    label='Duration (seconds)',
                    info='Snapped to the nearest valid frame count (17*n+5 at 24 fps).',
                )
                steps = gr.Slider(
                    10, 40, value=28, step=1,
                    label='Inference steps',
                    info='More steps = higher quality but slower. 28 is the default.',
                )
                seed = gr.Number(
                    value=42,
                    label='Seed',
                    info='Different seeds = different videos.',
                    precision=0,
                )
                rewrite_prompt = gr.Checkbox(
                    value=False,
                    label='Rewrite prompt (remote)',
                    info='Ask the conditioner Space to rewrite the prompt into MiniMax-H3 trained format.',
                )
            status_box = gr.Textbox(
                label='Status', interactive=False, lines=3,
                placeholder='Awaiting generation...',
            )

        with gr.Column(scale=2):
            output_video = gr.Video(label='Video + soundtrack', height=500)
            report_box = gr.Markdown(visible=False)
            with gr.Accordion('Downloads', open=False):
                dl_video = gr.File(label='Download .mp4')

    # --- Event wiring ---
    def generate(prompt_text, img_path, last_img_path, canvas_label,
                  dur, n_steps, s, rewrite, progress=gr.Progress(track_tqdm=True)):
        try:
            if not prompt_text or not prompt_text.strip():
                raise gr.Error('A prompt is required.')
            progress(0.0, desc='Conditioning ...')
            out_path, report = generate_video(
                prompt=prompt_text,
                image_path=img_path if img_path else None,
                last_image_path=last_img_path if last_img_path else None,
                canvas=canvas_label,
                duration=dur,
                steps=n_steps,
                seed=int(s),
                rewrite_prompt=rewrite,
                verbose=True,
            )
            report_md = (
                f'`{report["width"]}x{report["height"]}`, {report["num_frames"]} frames '
                f'({report["num_frames"]/FPS:.1f}s), {report["steps"]} steps · '
                f'denoise {report["generate_seconds"]:.0f}s · seed {report["seed"]}'
            )
            if report.get('refined_prompt'):
                report_md += f'\n\n**Refined prompt:** {report["refined_prompt"]}'
            free_cuda()
            return (
                gr.update(value=out_path),
                gr.update(value=report_md, visible=True),
                gr.update(value=out_path),
            )
        except Exception as e:
            traceback.print_exc(limit=4)
            raise gr.Error(f'Generation failed: {e}')

    btn_generate.click(
        generate,
        inputs=[prompt, image, last_image, canvas, duration, steps, seed, rewrite_prompt],
        outputs=[output_video, report_box, dl_video],
    )

    def _welcome():
        return (
            'Enter a prompt, optionally upload first/last frame images, '
            'then click "Generate video". First run loads + INT4-quantizes '
            'the model (~5-10 min after STEP 2).'
        )
    demo.load(_welcome, inputs=None, outputs=[status_box])

# --- Queue + launch ────────────────────────────────────────────────────
demo.queue(default_concurrency_limit=1, max_size=4)
try:
    from IPython.display import clear_output
    clear_output()
    clear_output(wait=True)
except Exception:
    pass
demo.launch(share=False, server_name='0.0.0.0', server_port=7860, show_error=True, height=1100)


In [ ]:
#@title STEP 5 — Keep alive + session summary
"""Standard AEI-suite keep-alive cell."""
import os, sys, time, pathlib
import IPython
from IPython.display import display, Javascript

print('='*72)
print('Keep-alive timer started.')
print('='*72)

try:
    summary = {
        'cache_root'    : str(drive_root),
        'ckpt_dir'      : str(CKPT_DIR),
        'out_dir'       : str(OUT_DIR),
        'torch'         : torch.__version__,
        'cuda'          : torch.version.cuda,
        'diffusers'     : diffusers.__version__,
        'transformers'  : transformers.__version__,
        'gpu'           : None,
    }
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        summary['gpu'] = f'{p.name}  ({p.total_memory / (1024**3):.1f} GB)'
    print('\n  Session summary')
    print('  ' + '-'*68)
    for k, v in summary.items():
        print(f'  {k:18s}: {v}')
except Exception as e:
    print(f'  WARN: summary print failed: {e}')

display(Javascript('''
function ClickConnect() {
  console.log("Keeping Colab alive — ", new Date().toLocaleTimeString());
  document.querySelector("colab-connect-button")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print('\n  Keep-alive timer registered (60 s interval).')


In [ ]:
#@title STEP 6 — Quick test (single video generation)
"""Stand-alone test. Generates one video with default parameters."""
import os, sys, time, pathlib
from IPython.display import display, FileLink

print('='*72)
print('MiniMax-H3 — single-video quick test')
print('='*72)

PROMPT = 'A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot'  #@param {type:"string"}
CANVAS = "960x544 · 16:9 fast"  #@param ['960x544 · 16:9 fast', '1024x576 · 16:9 fast', '1344x768 · 16:9 full', '544x960 · 9:16 fast', '640x1152 · 9:16', '544x544 · 1:1 fast', '768x768 · 1:1 full']
DURATION = 5  #@param {type:"slider", min:2, max:14, step:1}
STEPS = 28  #@param {type:"slider", min:10, max:40, step:1}
SEED = 42  #@param {type:"integer"}
REWRITE = False  #@param {type:"boolean"}

print(f'  Prompt   : {PROMPT[:60]}...')
print(f'  Canvas   : {CANVAS}')
print(f'  Duration : {DURATION}s')
print(f'  Steps    : {STEPS}')
print(f'  Seed     : {SEED}')
print()

t0 = time.time()
out_path, report = generate_video(
    prompt=PROMPT,
    canvas=CANVAS,
    duration=DURATION,
    steps=STEPS,
    seed=SEED,
    rewrite_prompt=REWRITE,
    verbose=True,
)
total_time = time.time() - t0

print()
print('='*72)
print(f'Video generated in {total_time:.0f}s')
print(f'  Resolution : {report["width"]}x{report["height"]}')
print(f'  Frames     : {report["num_frames"]} ({report["num_frames"]/FPS:.1f}s)')
print(f'  Denoise    : {report["generate_seconds"]:.0f}s')
print(f'  Output     : {out_path}')
sz = os.path.getsize(out_path) / 1024 / 1024
print(f'  Size       : {sz:.1f} MB')
print('='*72)

display(FileLink(out_path, result_html_prefix='Download video: '))
free_cuda()
print('\nSTEP 6 complete. Open the Gradio UI (STEP 4) for the full experience.')


In [ ]:
#@title STEP 7 — Batch generation (multiple prompts)
"""
Generate multiple videos from a list of prompts. Each video is saved as .mp4
with synchronized audio. The model is loaded once and reused.
"""
import os, sys, time, json, pathlib, traceback
import torch
from IPython.display import display, FileLink

print('='*72)
print('MiniMax-H3 — Batch generation')
print('='*72)

#@markdown Enter prompts (one per line, # comments OK)
PROMPTS = """A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot
A busy night market, neon signs reflecting in puddles, sizzling street food
A cellist playing a slow melody in an empty concert hall
Ocean waves crashing on a rocky shore at sunset, seagulls calling"""

CANVAS = "960x544 · 16:9 fast"  #@param ['960x544 · 16:9 fast', '1024x576 · 16:9 fast', '1344x768 · 16:9 full', '544x960 · 9:16 fast', '544x544 · 1:1 fast']
DURATION = 5  #@param {type:"slider", min:2, max:14, step:1}
STEPS = 28  #@param {type:"slider", min:10, max:40, step:1}
BASE_SEED = 42  #@param {type:"integer"}
REWRITE = False  #@param {type:"boolean"}

prompts = [l.strip() for l in PROMPTS.split('\n') if l.strip() and not l.startswith('#')]
print(f'  Prompts  : {len(prompts)}')
print(f'  Canvas   : {CANVAS}')
print(f'  Duration : {DURATION}s')
print(f'  Steps    : {STEPS}')
print()

output_subdir = OUT_DIR / f'batch_{int(time.time())}'
output_subdir.mkdir(parents=True, exist_ok=True)
batch_log = output_subdir / 'batch_log.jsonl'
log_f = open(batch_log, 'a', buffering=1)

results = []
batch_start = time.time()

for i, prompt_text in enumerate(prompts, 1):
    seed = BASE_SEED + i
    print(f'  [{i:03d}/{len(prompts)}] {prompt_text[:60]}...')
    t0 = time.time()
    try:
        out_path, report = generate_video(
            prompt=prompt_text,
            canvas=CANVAS,
            duration=DURATION,
            steps=STEPS,
            seed=seed,
            rewrite_prompt=REWRITE,
            verbose=False,
        )
        elapsed = time.time() - t0
        sz = os.path.getsize(out_path) / 1024 / 1024
        print(f'    OK ({elapsed:.0f}s, {sz:.1f} MB)')
        results.append(('ok', prompt_text, out_path))
        log_f.write(json.dumps({
            'idx': i, 'prompt': prompt_text, 'status': 'ok',
            'elapsed_s': elapsed, 'video': out_path,
            'width': report['width'], 'height': report['height'],
            'frames': report['num_frames'], 'size_mb': sz,
        }) + '\n')
    except Exception as e:
        elapsed = time.time() - t0
        print(f'    FAIL ({elapsed:.0f}s): {e}')
        traceback.print_exc(limit=2)
        results.append(('error', prompt_text, str(e)))
        log_f.write(json.dumps({
            'idx': i, 'prompt': prompt_text, 'status': 'error',
            'error': str(e), 'elapsed_s': elapsed,
        }) + '\n')
    free_cuda()

log_f.close()
total_elapsed = time.time() - batch_start
n_ok = sum(1 for r in results if r[0] == 'ok')
n_err = sum(1 for r in results if r[0] == 'error')

print()
print('='*72)
print(f'Batch complete: {n_ok} ok / {n_err} errors in {total_elapsed:.0f}s')
print(f'  Output: {output_subdir}')
print(f'  Log:   {batch_log}')
print('='*72)

for f in sorted(output_subdir.rglob('*.mp4')):
    sz = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:>30s}  {sz:>8.1f} MB')

if n_ok > 0:
    print(f'\n  Tip: open the videos in any player (they include audio).')
